# Agentic AI — Architectures & Patterns

AI agents are autonomous programs where **LLM outputs control the workflow** rather than simply producing text. The LLM's decisions determine what happens next.

**Traditional LLM:** Input → Output (one shot)  
**Agent:** Input → Think → Act → Observe → Repeat → Output

### Key Characteristics

| Characteristic | What it means |
|----------------|---------------|
| **Multiple LLM calls** | Decompose tasks; each call handles a focused sub-task |
| **Tool usage** | Call functions, APIs, or databases |
| **Interactive environment** | Observe state changes and adapt behavior |
| **Planning** | Decompose goals; manage step dependencies |
| **Autonomy** | Self-directed with minimal human intervention |

### The Agent Loop

![Agent Loop](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_agent_loop.png)

The agent continuously: receives input → makes an LLM call (think + decide) → takes an action on the environment → gets feedback → repeats until a STOP condition is met.

---

## Workflows vs Agents

**The core question:** Who is in control — your code or the LLM?

| | Workflows | Agents |
|---|-----------|--------|
| **Control** | Your code decides what happens next | LLM decides what happens next |
| **Execution path** | Fixed, predefined at design time | Dynamic, chosen at runtime |
| **Predictability** | Deterministic | Non-deterministic |
| **Debugging** | Easy — read the code | Requires tracing actual execution |
| **Use when** | Steps are known, costs must be controlled | Problem is exploratory, path is unknown |

> **Start with workflows.** Add agent behavior only where you truly need the flexibility — workflows are easier to build, test, and maintain.

## Workflow Design Patterns

Five proven templates for building deterministic AI workflows. Think of them as reusable blueprints — each solves a specific class of problem.

---

### Pattern 1: Prompt Chaining

Task decomposition into sequential sub-tasks where each LLM call receives the previous output as input. Each step uses a prompt optimized for that specific sub-task.

```
Input → [LLM 1] → Output 1 → [LLM 2] → Output 2 → [LLM 3] → Final Output
```

**Use when:** The task has a clear sequential structure and each step benefits from a focused, specialized prompt. Easy to debug — inspect any intermediate output.

![Pipeline](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_pipeline.png)

In [ ]:
from openai import OpenAI
client = OpenAI()

def chain_prompts(initial_input: str) -> str:
    """Sequential LLM calls — each output feeds the next."""

    # Step 1: Extract key business requirements
    step1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract the key business requirements as bullet points."},
            {"role": "user",   "content": initial_input}
        ]
    ).choices[0].message.content

    # Step 2: Design a solution from requirements
    step2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Design a technical solution for these requirements."},
            {"role": "user",   "content": step1}
        ]
    ).choices[0].message.content

    # Step 3: Write implementation plan from solution
    step3 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Write a detailed implementation plan."},
            {"role": "user",   "content": step2}
        ]
    ).choices[0].message.content

    return step3

### Pattern 2: Routing

Classify the input and direct it to the most appropriate specialist handler. The router can be an LLM, a rule-based classifier, or a hybrid.

```
Input → [Router] → Route A → [Specialist A] → Output
                 ↘ Route B → [Specialist B]
                 ↘ Route C → [Specialist C]
```

**Use when:** Inputs fall into distinct categories (e.g., technical / billing / general) that benefit from different prompts, models, or processing logic.

![Router](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_router.png)

In [ ]:
from openai import OpenAI
client = OpenAI()

# Specialist system prompts
SPECIALISTS = {
    "technical": "You are a technical expert. Answer technical questions about code and software.",
    "billing":   "You are a billing specialist. Handle payments, invoices, and subscription questions.",
    "general":   "You are a helpful general assistant.",
}

def route_and_answer(user_query: str) -> str:
    """Route query to the right specialist via an LLM router."""

    # Router LLM decides which specialist to use
    routing = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Classify this query into exactly one category: technical, billing, or general.
Reply with only the category word.

Query: {user_query}"""
        }]
    ).choices[0].message.content.strip().lower()

    specialist = SPECIALISTS.get(routing, SPECIALISTS["general"])
    print(f"Routed to: {routing}")

    # Specialist LLM answers
    answer = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": specialist},
            {"role": "user",   "content": user_query}
        ]
    ).choices[0].message.content

    return answer

# print(route_and_answer("My invoice shows wrong amount"))
# print(route_and_answer("How do I implement a binary search tree?"))

### Pattern 3: Parallelization (Coordinator + Aggregator)

Split independent sub-tasks across concurrent LLM calls, then aggregate the results. Reduces wall-clock time to the duration of the single longest sub-task.

```
Input → [Coordinator] → [LLM 1 ‖ LLM 2 ‖ LLM 3] → [Aggregator] → Output
                         (all run concurrently)
```

**Use when:** Sub-tasks are independent of each other and latency matters. Common for multi-document analysis, batch translation, or parallel perspective gathering.

![Parallelization](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_parallelization.png)

In [ ]:
import asyncio
from openai import OpenAI
client = OpenAI()

async def parallel_analysis(topic: str) -> dict:
    """Run multiple analyses simultaneously, then aggregate."""

    async def analyze(perspective: str) -> str:
        response = await asyncio.to_thread(
            client.chat.completions.create,
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": f"Analyze from a {perspective} perspective in 2-3 sentences."},
                {"role": "user",   "content": topic}
            ]
        )
        return response.choices[0].message.content

    # Coordinator: launch all analyses in parallel
    technical, business, risk = await asyncio.gather(
        analyze("technical"),
        analyze("business"),
        analyze("risk"),
    )

    # Aggregator: synthesize results
    combined = f"Technical: {technical}\n\nBusiness: {business}\n\nRisk: {risk}"
    summary = await asyncio.to_thread(
        client.chat.completions.create,
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Synthesize these perspectives into a concise executive summary."},
            {"role": "user",   "content": combined}
        ]
    )

    return {
        "technical": technical,
        "business":  business,
        "risk":      risk,
        "summary":   summary.choices[0].message.content
    }

# result = asyncio.run(parallel_analysis("Migrating a monolith to microservices"))
# print(result["summary"])

### Pattern 4: Orchestrator-Workers

An orchestrator LLM dynamically plans and decomposes a complex task into sub-tasks, delegates them to worker LLMs, then synthesizes the results into a final output.

```
Task → [Orchestrator] ──── plans + delegates ────→ [Worker 1 ‖ Worker 2 ‖ Worker 3]
            ↑                                                      ↓
            └─────────────── synthesizes results ─────────────────┘
```

**Use when:** The sub-tasks aren't known upfront and require dynamic planning. The orchestrator can also replan based on what workers discover.

![Orchestrator](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_orchestrator.png)

In [ ]:
import asyncio, json
from openai import OpenAI
client = OpenAI()

async def orchestrate_task(complex_task: str) -> str:
    """Orchestrator breaks task into sub-tasks, workers execute, synthesizer combines."""

    # Orchestrator: decompose task
    plan_response = await asyncio.to_thread(
        client.chat.completions.create,
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Break this task into exactly 3 parallel sub-tasks.
Return JSON: {{"subtasks": ["subtask1", "subtask2", "subtask3"]}}

Task: {complex_task}"""
        }]
    )
    plan = json.loads(plan_response.choices[0].message.content)
    subtasks = plan["subtasks"]
    print(f"Orchestrator created {len(subtasks)} sub-tasks")

    # Workers: execute sub-tasks in parallel
    async def worker(subtask: str) -> str:
        resp = await asyncio.to_thread(
            client.chat.completions.create,
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a specialist. Complete your assigned sub-task thoroughly."},
                {"role": "user",   "content": subtask}
            ]
        )
        return resp.choices[0].message.content

    results = await asyncio.gather(*[worker(st) for st in subtasks])

    # Synthesizer: combine worker results
    combined = "\n\n".join(f"Sub-task {i+1} result:\n{r}" for i, r in enumerate(results))
    synthesis = await asyncio.to_thread(
        client.chat.completions.create,
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Synthesize these sub-task results into a cohesive final answer."},
            {"role": "user",   "content": f"Original task: {complex_task}\n\n{combined}"}
        ]
    )
    return synthesis.choices[0].message.content

# result = asyncio.run(orchestrate_task("Write a comprehensive market analysis for electric vehicles"))
# print(result)

### Pattern 5: Evaluator-Optimizer (Generator-Evaluator)

One LLM generates output; another evaluates and critiques it. If rejected, the feedback is fed back to the generator. The loop continues until the quality threshold is met or max retries are exhausted.

```
Input → [Generator] → Draft → [Evaluator] → Pass → Output
              ↑                      ↓ Fail
              └────── feedback ───────┘
```

**Use when:** First-attempt outputs are often insufficient and iterative refinement yields meaningful quality improvement — code generation, content editing, data extraction.

![Evaluator](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_evaluator.png)

In [ ]:
from pydantic import BaseModel
from openai import OpenAI
client = OpenAI()

class EvalResult(BaseModel):
    score: int       # 1–10
    passed: bool
    feedback: str

def generate_with_quality_gate(task: str, threshold: int = 7, max_retries: int = 3) -> str:
    """Generator-Evaluator loop: retry until quality threshold is met."""

    for attempt in range(1, max_retries + 1):
        # Generator LLM
        content = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": task}]
        ).choices[0].message.content

        # Evaluator LLM (structured output)
        eval_resp = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": f"Rate this response quality 1-10.\nTask: {task}\nResponse: {content}"
            }],
            response_format=EvalResult
        )
        result = eval_resp.choices[0].message.parsed
        print(f"Attempt {attempt}: score={result.score}, passed={result.passed}")

        if result.passed and result.score >= threshold:
            return content  # Accepted

        # Rejected: incorporate feedback for next attempt
        task = f"{task}\n\nPrevious feedback: {result.feedback}. Please improve."

    return content  # Return best attempt after max retries

### Pattern 6: No-Framework Agentic Loop

You don't need a framework to implement agentic patterns. Direct API calls with a generator-evaluator loop demonstrate the core concepts clearly.

The example below builds a personal AI chatbot with:
- A **generator** (OpenAI) that answers as a persona
- An **evaluator** (Gemini) that checks quality
- A **re-runner** that regenerates with feedback if quality fails

In [ ]:
import os
import gradio as gr
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv(override=True)

openai_client = OpenAI()
gemini_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

NAME = "Alex"
BACKGROUND = "Alex is a senior software engineer with 10 years of experience in Python and cloud systems."

system_prompt = f"""You are acting as {NAME}.
Answer questions about {NAME}'s career, skills, and experience.
Be professional and engaging.

## Background:
{BACKGROUND}
"""

def evaluate(reply: str, message: str, history: list) -> Evaluation:
    """Evaluator LLM (Gemini) checks if the reply is acceptable."""
    eval_prompt = f"""You evaluate if an AI agent's response is acceptable quality.
The agent is playing the role of {NAME} on their personal website.

Context: {BACKGROUND}

User message: {message}
Agent reply: {reply}

Is this reply acceptable? Provide structured feedback."""

    response = gemini_client.beta.chat.completions.parse(
        model="gemini-2.0-flash",
        messages=[{"role": "user", "content": eval_prompt}],
        response_format=Evaluation
    )
    return response.choices[0].message.parsed

def rerun(reply: str, message: str, history: list, feedback: str) -> str:
    """Regenerate with evaluator feedback injected into system prompt."""
    improved_system = system_prompt + f"""

## Quality control rejected your previous answer
Your attempted answer: {reply}
Reason for rejection: {feedback}
Please improve your response."""

    messages = [{"role": "system", "content": improved_system}] + history + [{"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

def chat(message: str, history: list) -> str:
    """Main chat function: generate → evaluate → rerun if needed."""
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    # Generator
    reply = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages).choices[0].message.content

    # Evaluator
    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Passed quality check")
    else:
        print(f"Quality check failed: {evaluation.feedback}")
        reply = rerun(reply, message, history, evaluation.feedback)

    return reply

# gr.ChatInterface(chat, type="messages").launch()

## Pattern Selection Guide

| Pattern | Use when |
|---------|----------|
| **Prompt Chaining** | Steps are sequential; each needs focused prompting |
| **Routing** | Inputs fall into distinct categories |
| **Parallelization** | Sub-tasks are independent and latency matters |
| **Orchestrator-Workers** | Task is complex; sub-tasks aren't known upfront |
| **Evaluator-Optimizer** | Quality matters and iteration improves results |

Patterns can be combined: Orchestrator-Workers + Parallelization (workers run concurrently), Routing + Prompt Chaining (each route is its own chain).

> These patterns give you the benefits of AI with **predictable, auditable behavior**. Start here before reaching for a full agent framework.

## True Agents — Dynamic Execution

In workflows, **your code** decides what happens next. In agents, **the LLM** decides.

### Agent Characteristics

- **Open-ended execution:** No predetermined path — the agent explores the problem space dynamically.
- **Feedback loops:** Observe → Act → Reflect → Adjust, cycling until the goal is met.
- **No fixed path:** Control flow determined at runtime; the agent can backtrack or change strategy mid-execution.

### When to Use Agents

- Problem is exploratory (root cause of a bug, path through unknown data)
- Solution path is unknown upfront (web research, open-ended analysis)
- Tasks require adaptive or creative behavior
- A workflow would require too many explicit branches to code

---

## Risks & Mitigations

| Risk | Why it happens | Mitigation |
|------|---------------|------------|
| **Unpredictable path** | LLM decisions are non-deterministic | Log every decision; set `temperature=0` in tests |
| **Inconsistent output** | LLM format varies between runs | Enforce structured outputs (Pydantic); validate before returning |
| **Runaway cost** | Retry loops and exploration multiply calls | Set `max_iterations`, token budgets, and dollar caps |
| **No visibility** | Execution trace not recorded | Emit structured logs; build dashboards; set alerts on error rate and cost |

---

## Guardrails

Programmatic constraints that keep agents operating within safe bounds — applied at input, during execution, and on output.

```python
class GuardedAgent:
    def run(self, user_input: str):
        # 1. Input guardrails — length, content filtering, format checks
        validated_input = self.input_guardrails(user_input)

        # 2. Execution guardrails — max iterations, cost cap, allowed tools
        with self.resource_limits():
            with self.tool_restrictions():
                output = self.agent.run(validated_input)

        # 3. Output guardrails — PII redaction, toxicity check, structure validation
        return self.output_guardrails(output)
```

**Rule of thumb:** Start restrictive. Relax guardrails incrementally as you gain confidence in your agent's behavior.

## Framework Landscape

Sorted from simple to complex. Start with the simplest that fits your use case.

---

### 1. Raw API Calls
Direct LLM calls, no abstraction. Write every prompt, loop, and tool handler yourself.
- **Best for:** Learning how agents work, simple tasks, fully custom requirements
- **Trade-off:** Most time-consuming; no built-in patterns

---

### 2. OpenAI Agents SDK
Minimal multi-agent framework built around Agents, Handoffs, and Guardrails — stays close to the OpenAI API.
- **Best for:** OpenAI-only projects, quick prototypes, good built-in tracing
- **Trade-off:** OpenAI-specific; limited to moderate complexity

---

### 3. LangGraph
State-machine framework — define agent workflows as directed graphs of nodes (actions) and edges (transitions).
- **Best for:** Complex workflows with branching logic, explicit state management, visual debugging
- **Trade-off:** Steeper learning curve; over-engineering for simple tasks

---

### 4. CrewAI
Role-based framework — agents have roles, goals, and backstories; they collaborate as a crew on tasks.
- **Best for:** Content pipelines, research workflows with clear role separation
- **Trade-off:** More opinionated; abstraction can hide what's happening

---

### 5. AutoGen (Microsoft)
Agents collaborate through conversation — they can debate, critique, and refine each other's work.
- **Best for:** Complex multi-agent reasoning, critique loops, research-oriented tasks
- **Trade-off:** Most expensive (many LLM calls); hardest to debug and control

---

### 6. MCP (Model Context Protocol)
A **protocol**, not a framework — standardizes how LLMs connect to tools and data sources (like USB for AI).
- **Best for:** Reusable tool integrations across different projects and apps
- **Trade-off:** Does not provide agent orchestration logic — you still need a framework or raw API

---

## Selection Guide

| Need | Start with |
|------|-----------|
| Learning the fundamentals | Raw API |
| Quick OpenAI prototype | OpenAI Agents SDK |
| Complex branching state machine | LangGraph |
| Role-based team workflows | CrewAI |
| Multi-agent conversational reasoning | AutoGen |
| Reusable standardized tool integrations | MCP |

**Decision rule:** Start with the simplest option that could work. The cost of adding complexity must be justified by a clear benefit you can't get any other way.

## Asyncio — Concurrent Execution

All agent frameworks use async Python. When a coroutine is waiting (e.g., for an API response), the event loop runs other coroutines instead of blocking — making concurrent I/O fast without threads.

```python
async def do_something() -> str:
    return "done"

result = await do_something()        # inside an async context
# asyncio.run(do_something())        # from synchronous code
```

### Sequential vs Concurrent

```python
# Sequential: 3 + 2 = 5 seconds total
async def slow():
    coffee = await make_coffee()     # wait 3s, then...
    toast  = await toast_bread()     # wait 2s
    return coffee, toast

# Concurrent: max(3, 2) = 3 seconds total
async def fast():
    coffee, toast = await asyncio.gather(
        make_coffee(),               # both start
        toast_bread()               # at the same time
    )
    return coffee, toast
```

### Common Patterns

```python
# 1. gather — run tasks concurrently, collect all results
results = await asyncio.gather(task1(), task2(), task3())

# 2. wait_for — cancel if too slow
result = await asyncio.wait_for(slow_op(), timeout=30)

# 3. to_thread — run blocking (sync) code without blocking the event loop
result = await asyncio.to_thread(blocking_cpu_function)
```

### Why Agent Frameworks Use It

LLM calls, web searches, and database queries are all I/O-bound. Running them concurrently means a 3-call agent step takes as long as the slowest single call — not 3× longer.

**Use asyncio when:** waiting for network I/O (APIs, databases, web scraping)  
**Use multiprocessing when:** CPU-intensive computation (training, image processing)